# Protein prediction, samples and templates

Predict a short protein, inspect confidence, and reuse a structure as a template. An optional output mount lets you inspect full PAE/PDE matrices.

Tested with **Boltz2 NIM 1.9.0 / client 0.6.0**.
From the repository root, install `python -m pip install -e ".[notebooks]"`
and start `jupyter lab examples/notebooks` in that environment.
Use a [ready NIM server](../../docs/deployment.md); `BOLTZ2_NIM_URL`
defaults to `http://localhost:8000`. Run cells in order.

[Setup and output files](../README.md#notebooks) ·
[NIM release notes](https://docs.nvidia.com/nim/bionemo/boltz2/1.9.0/release-notes.html)

Copyright (c) 2025-2026 NVIDIA Corporation.

In [ ]:
import os
from pathlib import Path
import boltz2_client
from boltz2_client import Boltz2Client, Polymer, PredictionRequest, StructuralTemplate

client = Boltz2Client(
    base_url=os.getenv("BOLTZ2_NIM_URL", "http://localhost:8000"), timeout=1200,
)
metadata = await client.get_service_metadata()
print(f"Client {boltz2_client.__version__}; connected NIM {metadata.version} (tested: 1.9.0)")
output_dir = Path("notebook_outputs") / "protein"
output_dir.mkdir(parents=True, exist_ok=True)

## Predict two samples

This short sequence demonstrates request syntax. `diffusion_samples=2` requests
two structures; `max_parallel_samples=1` processes one sample at a time to reduce
peak GPU memory. Model confidence does not establish experimental accuracy.

For matrix files, set `BOLTZ2_SERVER_OUTPUT_DIR` to the accessible host directory
mounted at `/opt/nim/output` **before running this cell**. Without that setting,
the example requests structures and aggregate scores only.

In [ ]:
from uuid import uuid4

sequence = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
server_output = os.getenv("BOLTZ2_SERVER_OUTPUT_DIR")
request = PredictionRequest(
    polymers=[Polymer(id="A", molecule_type="protein", sequence=sequence)],
    diffusion_samples=2,
    max_parallel_samples=1,
    complex_id="protein_" + uuid4().hex[:12],
    write_full_pae=bool(server_output),
    write_full_pde=bool(server_output),
)
result = await client.predict(request, output_dir=output_dir / "samples")
for index, confidence in enumerate(result.confidence_scores):
    print(f"structure_{index}.cif: confidence={confidence:.3f}")
print("Saved structures and JSON:", (output_dir / "samples").resolve())

## View the first sample

The inline viewer uses py3Dmol and loads its JavaScript from a public CDN.
Trust an executed notebook to display its viewer. The saved mmCIF files can
also be opened in a desktop molecular viewer.

In [ ]:
import py3Dmol

view = py3Dmol.view(width=800, height=450)
view.addModel(result.structures[0].structure, "cif")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

## Use a structural template

This reuses the first prediction to demonstrate the template API; it is not
independent structural evidence. Supply an appropriate experimental template
and its chain ID for a scientific application.

In [ ]:
template_request = PredictionRequest(
    polymers=[Polymer(
        id="A", molecule_type="protein", sequence=sequence,
        structural_templates=[StructuralTemplate(
            structure=result.structures[0].structure, format="cif", chain_id="A",
        )],
    )],
)
template_result = await client.predict(template_request, output_dir=output_dir / "template")
print("Template-conditioned confidence:", template_result.confidence_scores)

## Read optional matrix files

In NIM 1.9.0, `pae` and `pde` in the JSON response stay `null`. Full matrices
are server-side NPZ files; the client does not download them. `complex_id`
names the output directory in the tested container. Aggregate PDE scores
remain in JSON. See [output paths](../../docs/outputs.md).

In [ ]:
print("JSON PAE/PDE:", result.pae, result.pde)
print("Aggregate PDE:", result.complex_pde_scores)
if server_output:
    import numpy as np

    for kind in ("pae", "pde"):
        matrix_dir = Path(server_output) / request.complex_id / kind
        files = sorted(matrix_dir.glob("*.npz"))
        if len(files) != len(result.structures):
            raise FileNotFoundError(f"Expected one {kind} file per sample in {matrix_dir}")
        for file in files:
            with np.load(file, allow_pickle=False) as archive:
                matrix = archive[kind]
                print(file.name, matrix.shape)
else:
    print("Matrix files were not requested; set BOLTZ2_SERVER_OUTPUT_DIR to enable them.")